# ABI vs CARBayes on Glasgow, California, and South Korea

This notebook loads the original trained ABI network from `Training/Checkpoints/poisson_dagar.keras`, reconstructs the empirical inputs for Glasgow, California, and South Korea, loads the corresponding `CARBayes` exports from `setup_and_diagnostics`, and compares the two methods under the median probability model rule. The main cross-method parameter comparisons are `alpha` versus `eta` for boundary strength and `tau2` versus `sigma2_w` for spatial variance scale.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

from pathlib import Path
import warnings

import bayesflow as bf
import geopandas as gpd
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_repository_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Training").is_dir() and (candidate / "Simulation Experiments").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the ABI_poisson_regression repository root.")


REPOSITORY_ROOT = find_repository_root()

PROJECT_DIR = REPOSITORY_ROOT / "Real Data Analysis"
DATA_DIR = PROJECT_DIR / "Data"
CHECKPOINTS_DIR = REPOSITORY_ROOT / "Training" / "Checkpoints"
CARBAYES_DIR = PROJECT_DIR / "setup_and_diagnostics"
COMPARISON_DIR = PROJECT_DIR / "results_ABI_vs_CARBayes"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = CHECKPOINTS_DIR / "poisson_dagar.keras"
ABI_NUM_SAMPLES = 100_000
PPC_NUM_SAMPLES = 1000
SCRIPT_PLOT_NUM_SAMPLES = 1000
PREDICTIVE_LAMBDA_MAX = 1e7
LOG_PREDICTIVE_LAMBDA_MIN = np.log(1e-2)
LOG_PREDICTIVE_LAMBDA_MAX = np.log(PREDICTIVE_LAMBDA_MAX)
THRESHOLD = np.log(2.0)
PPC_ORDERING_MODE = "identity"  # set to "random" to mimic the exploratory notebook more closely
MAP_BOUNDARY_LINEWIDTH = 2.5
MAP_LEGEND_SHRINK = 0.75
MAP_LEGEND_TICKSIZE = 18
MAP_LEGEND_LABELSIZE = 18

np.random.seed(123)
keras.utils.set_random_seed(123)
np.set_printoptions(suppress=True)

print("Project directory:", PROJECT_DIR)
print("Model path:", MODEL_PATH)
print("CARBayes exports:", CARBAYES_DIR)

In [ ]:
def _dagar_factors(A, rho, ordering):
    n = A.shape[0]
    rho2 = rho ** 2
    inv_order = np.argsort(ordering)

    B = np.zeros((n, n), dtype=np.float32)
    lam = np.zeros(n, dtype=np.float32)

    for pos in range(n):
        i = ordering[pos]
        preds = [ordering[q] for q in range(pos) if A[i, ordering[q]] == 1]
        n_lt = len(preds)
        denom = 1.0 + max(n_lt - 1, 0) * rho2
        b_val = rho / denom if n_lt > 0 else 0.0
        for j in preds:
            B[pos, inv_order[j]] = b_val
        lam[pos] = denom / (1.0 - rho2)

    ImB = np.eye(n, dtype=np.float32) - B
    return ImB, lam


def _repair_isolates_deterministic(A_filtered, A, Z):
    A_rep = A_filtered.copy().astype(np.float32)
    for i in range(A_rep.shape[0]):
        if A_rep[i].sum() == 0:
            neighbors = np.where(A[i] == 1)[0]
            if len(neighbors) > 0:
                j = neighbors[np.argmin(Z[i, neighbors])]
                A_rep[i, j] = 1.0
                A_rep[j, i] = 1.0
    return A_rep


def _masked_row_mean(values, mask):
    mask_f = mask.astype(np.float32)
    denom = mask_f.sum(axis=1)
    denom_safe = np.where(denom == 0, 1.0, denom)
    out = (values * mask_f).sum(axis=1) / denom_safe
    out[denom == 0] = 0.0
    return out.astype(np.float32)


def _safe_mean_1d(values):
    return float(values.mean()) if values.size > 0 else 0.0


def _safe_corr(x, y):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    x_c = x - x.mean()
    y_c = y - y.mean()
    denom = np.sqrt(np.mean(x_c ** 2) * np.mean(y_c ** 2))
    if denom < 1e-8:
        return 0.0
    return float(np.mean(x_c * y_c) / denom)


def _observed_eta_signal_metrics(y, e, A, Z, Z_median, r_lag_all=None):
    A_bool = A == 1
    edge_i, edge_j = np.where(np.triu(A_bool, 1))

    zero_metrics = dict(
        edge_absdiff_low=0.0,
        edge_absdiff_mid=0.0,
        edge_absdiff_high=0.0,
        edge_concord_low=0.0,
        edge_concord_mid=0.0,
        edge_concord_high=0.0,
        edge_absdiff_slope=0.0,
        edge_absdiff_gap=0.0,
        edge_concord_gap=0.0,
        edge_corr_all=0.0,
        lag_corr_all=0.0,
        lag_slope_all=0.0,
        edge_semivar_all=0.0,
        local_moran_mean=0.0,
    )
    if edge_i.size == 0:
        return zero_metrics

    r = (np.log(y + 0.5) - np.log(e)).astype(np.float32)
    z_rel = (Z / Z_median).astype(np.float32)

    z_edge = z_rel[edge_i, edge_j]
    absdiff_edge = np.abs(r[edge_i] - r[edge_j]).astype(np.float32)
    sqdiff_edge = ((r[edge_i] - r[edge_j]) ** 2).astype(np.float32)

    r_centered = (r - r.mean()).astype(np.float32)
    var_r = float(np.mean(r_centered ** 2))
    var_r_safe = max(var_r, 1e-8)
    concord_edge = (r_centered[edge_i] * r_centered[edge_j]).astype(np.float32)

    low = z_edge <= 0.75
    mid = (z_edge > 0.75) & (z_edge <= 1.25)
    high = z_edge > 1.25

    z_bar = float(z_edge.mean())
    absdiff_bar = float(absdiff_edge.mean())
    var_z = float(np.mean((z_edge - z_bar) ** 2))

    if var_z < 1e-8:
        edge_absdiff_slope = 0.0
    else:
        edge_absdiff_slope = float(
            np.mean((z_edge - z_bar) * (absdiff_edge - absdiff_bar)) / var_z
        )

    if r_lag_all is None:
        degree = A_bool.sum(axis=1).astype(np.float32)
        degree_safe = np.where(degree == 0, 1.0, degree)
        W = A / degree_safe[:, None]
        r_lag_all = (W @ r).astype(np.float32)

    r_lag_centered = (r_lag_all - r_lag_all.mean()).astype(np.float32)
    lag_slope_all = float(np.mean(r_centered * r_lag_centered) / var_r_safe)
    lag_corr_all = _safe_corr(r, r_lag_all)
    local_moran = (r_centered * r_lag_all / var_r_safe).astype(np.float32)
    edge_corr_all = float(np.mean(concord_edge) / var_r_safe)
    edge_semivar_all = float(0.5 * np.mean(sqdiff_edge))

    return dict(
        edge_absdiff_low=_safe_mean_1d(absdiff_edge[low]),
        edge_absdiff_mid=_safe_mean_1d(absdiff_edge[mid]),
        edge_absdiff_high=_safe_mean_1d(absdiff_edge[high]),
        edge_concord_low=_safe_mean_1d(concord_edge[low]),
        edge_concord_mid=_safe_mean_1d(concord_edge[mid]),
        edge_concord_high=_safe_mean_1d(concord_edge[high]),
        edge_absdiff_slope=edge_absdiff_slope,
        edge_absdiff_gap=_safe_mean_1d(absdiff_edge[high]) - _safe_mean_1d(absdiff_edge[low]),
        edge_concord_gap=_safe_mean_1d(concord_edge[low]) - _safe_mean_1d(concord_edge[high]),
        edge_corr_all=edge_corr_all,
        lag_corr_all=lag_corr_all,
        lag_slope_all=lag_slope_all,
        edge_semivar_all=edge_semivar_all,
        local_moran_mean=float(local_moran.mean()),
    )


def _build_observed_features(x, y, e, A, Z, Z_median, M):
    A_bool = A == 1

    x = x.astype(np.float32)
    y = y.astype(np.float32)
    e = e.astype(np.float32)

    log_y = np.log1p(y).astype(np.float32)
    log_e = np.log(e).astype(np.float32)
    r = (np.log(y + 0.5) - log_e).astype(np.float32)

    r_centered = (r - r.mean()).astype(np.float32)
    var_r_safe = max(float(np.mean(r_centered ** 2)), 1e-8)

    degree = A_bool.sum(axis=1).astype(np.float32)
    degree_safe = np.where(degree == 0, 1.0, degree)

    z_rel = (Z / Z_median).astype(np.float32)
    neigh_r = np.broadcast_to(r[None, :], z_rel.shape).astype(np.float32)
    abs_r_diff = np.abs(r[:, None] - r[None, :]).astype(np.float32)

    low_mask = A_bool & (z_rel <= 0.75)
    mid_mask = A_bool & (z_rel > 0.75) & (z_rel <= 1.25)
    high_mask = A_bool & (z_rel > 1.25)

    r_lag_all = _masked_row_mean(neigh_r, A_bool)
    absdiff_all = _masked_row_mean(abs_r_diff, A_bool)
    r_lag_low = _masked_row_mean(neigh_r, low_mask)
    r_lag_mid = _masked_row_mean(neigh_r, mid_mask)
    r_lag_high = _masked_row_mean(neigh_r, high_mask)
    absdiff_low = _masked_row_mean(abs_r_diff, low_mask)
    absdiff_mid = _masked_row_mean(abs_r_diff, mid_mask)
    absdiff_high = _masked_row_mean(abs_r_diff, high_mask)
    prop_low = (low_mask.sum(axis=1) / degree_safe).astype(np.float32)
    prop_mid = (mid_mask.sum(axis=1) / degree_safe).astype(np.float32)
    prop_high = (high_mask.sum(axis=1) / degree_safe).astype(np.float32)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean_z_rel = np.nanmean(np.where(A_bool, z_rel, np.nan), axis=1)
        max_z_rel = np.nanmax(np.where(A_bool, z_rel, np.nan), axis=1)

    mean_z_rel = np.nan_to_num(mean_z_rel, nan=0.0).astype(np.float32)
    max_z_rel = np.nan_to_num(max_z_rel, nan=0.0).astype(np.float32)

    edge_metrics = _observed_eta_signal_metrics(
        y=y,
        e=e,
        A=A,
        Z=Z,
        Z_median=Z_median,
        r_lag_all=r_lag_all,
    )

    local_moran = (r_centered * r_lag_all / var_r_safe).astype(np.float32)
    local_semivar = (((r - r_lag_all) ** 2) / var_r_safe).astype(np.float32)

    obs = np.stack(
        [
            x,
            log_y,
            log_e,
            r,
            degree,
            r_lag_all,
            absdiff_all,
            r_lag_low,
            r_lag_mid,
            r_lag_high,
            absdiff_low,
            absdiff_mid,
            absdiff_high,
            prop_low,
            prop_mid,
            prop_high,
            mean_z_rel,
            max_z_rel,
            local_moran,
            local_semivar,
            np.full(len(x), M, dtype=np.float32),
            np.full(len(x), edge_metrics["edge_absdiff_slope"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_absdiff_gap"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_concord_gap"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_corr_all"], dtype=np.float32),
            np.full(len(x), edge_metrics["lag_corr_all"], dtype=np.float32),
            np.full(len(x), edge_metrics["lag_slope_all"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_semivar_all"], dtype=np.float32),
        ],
        axis=-1,
    )

    return obs.astype(np.float32)


def prepare_real_dataset(config):
    gdf = gpd.read_file(DATA_DIR / config["gpkg"])
    A_real = pd.read_csv(DATA_DIR / config["adjacency_csv"]).values.astype(np.float32)
    area_ids = gdf[config["id_col"]].astype(str).to_numpy()

    x_real_raw = gdf[config["x_col"]].to_numpy(dtype=np.float32)
    x_real_std = (x_real_raw - x_real_raw.mean()) / (x_real_raw.std() + 1e-8)
    x_real = x_real_std.astype(np.float32)
    y_real = gdf[config["y_col"]].to_numpy(dtype=np.float32)
    e_real = gdf[config["e_col"]].to_numpy(dtype=np.float32)

    Z_real = np.abs(x_real[:, None] - x_real[None, :]).astype(np.float32)
    Z_edges_real = Z_real[A_real == 1]
    if Z_edges_real.size == 0:
        Z_median_real = 1.0
    else:
        Z_median_real = float(np.median(Z_edges_real) + 1e-8)

    M_real = float(np.log(2.0) / Z_median_real)
    obs_real = _build_observed_features(
        x=x_real,
        y=y_real,
        e=e_real,
        A=A_real,
        Z=Z_real,
        Z_median=Z_median_real,
        M=M_real,
    )

    return {
        "gdf": gdf,
        "A_real": A_real,
        "area_ids": area_ids,
        "x_real": x_real,
        "x_real_raw": x_real_raw,
        "y_real": y_real,
        "e_real": e_real,
        "Z_real": Z_real,
        "Z_median_real": Z_median_real,
        "M_real": M_real,
        "N_real": int(A_real.shape[0]),
        "obs_real": obs_real,
        "conditions": {
            "obs": obs_real[np.newaxis, ...],
            "N": np.array([A_real.shape[0]], dtype=np.int32),
        },
    }


def summarize_draws(samples, name):
    q025, q50, q975 = np.quantile(samples, [0.025, 0.5, 0.975])
    return {
        "parameter": name,
        "median": float(q50),
        "lower_95": float(q025),
        "upper_95": float(q975),
    }


def extract_abi_summary(post_draws, M_real):
    beta0_post = post_draws["beta"][0, :, 0]
    sigma2_post = post_draws["sigma2_w"][0, :, 0]
    eta_raw_post = post_draws["eta_raw"][0, :, 0]
    rho_post = post_draws["rho"][0, :, 0]
    eta_post = eta_raw_post * M_real

    rows = [
        summarize_draws(beta0_post, "beta_0"),
        summarize_draws(sigma2_post, "sigma2_w"),
        summarize_draws(eta_raw_post, "eta_raw"),
        summarize_draws(eta_post, "eta"),
        summarize_draws(rho_post, "rho"),
    ]
    return pd.DataFrame(rows)


def build_parameter_comparison(abi_summary, car_summary):
    abi_lookup = abi_summary.set_index("parameter")
    car_lookup = car_summary.set_index("parameter")

    comparison_spec = [
        {"comparison": "intercept", "abi_parameter": "beta_0", "car_parameter": "beta_0"},
        {"comparison": "variance_scale", "abi_parameter": "sigma2_w", "car_parameter": "tau2"},
        {"comparison": "boundary_strength", "abi_parameter": "eta", "car_parameter": "alpha"},
        {"comparison": "spatial_dependence", "abi_parameter": "rho", "car_parameter": None},
    ]

    rows = []
    for spec in comparison_spec:
        abi_row = abi_lookup.loc[spec["abi_parameter"]] if spec["abi_parameter"] in abi_lookup.index else None
        car_row = car_lookup.loc[spec["car_parameter"]] if spec["car_parameter"] in car_lookup.index else None

        rows.append({
            "comparison": spec["comparison"],
            "abi_parameter": spec["abi_parameter"],
            "car_parameter": spec["car_parameter"],
            "abi_median": float(abi_row["median"]) if abi_row is not None else np.nan,
            "abi_lower_95": float(abi_row["lower_95"]) if abi_row is not None else np.nan,
            "abi_upper_95": float(abi_row["upper_95"]) if abi_row is not None else np.nan,
            "car_median": float(car_row["median"]) if car_row is not None else np.nan,
            "car_lower_95": float(car_row["lower_95"]) if car_row is not None else np.nan,
            "car_upper_95": float(car_row["upper_95"]) if car_row is not None else np.nan,
        })

    return pd.DataFrame(rows)


def compute_abi_edge_table(post_draws, x_real, A_real, area_ids, M_real):
    rows, cols = np.where(A_real == 1)
    mask = rows < cols
    i_edges = rows[mask]
    j_edges = cols[mask]
    z_edges = np.abs(x_real[i_edges] - x_real[j_edges])
    eta_post = post_draws["eta_raw"][0, :, 0] * M_real

    boundary_draws = (z_edges[None, :] * eta_post[:, None]) > THRESHOLD
    boundary_prob = boundary_draws.mean(axis=0)

    return pd.DataFrame({
        "from_index0": i_edges.astype(int),
        "to_index0": j_edges.astype(int),
        "from_id": area_ids[i_edges],
        "to_id": area_ids[j_edges],
        "edge_dissimilarity": z_edges.astype(np.float32),
        "boundary_prob_abi": boundary_prob,
        "boundary_median_abi": (boundary_prob > 0.5).astype(int),
    })


def _get_ordering(n, mode="identity"):
    if mode == "random":
        return np.random.permutation(n).astype(int)
    return np.arange(n, dtype=int)


def compute_abi_predictive_summaries(post_draws, data, ppc_num_samples=PPC_NUM_SAMPLES, ordering_mode=PPC_ORDERING_MODE):
    n_draws = min(ppc_num_samples, post_draws["beta"].shape[1])
    N_real = data["N_real"]

    beta_samples = post_draws["beta"][0, :n_draws, 0]
    rho_samples = post_draws["rho"][0, :n_draws, 0]
    eta_samples = post_draws["eta_raw"][0, :n_draws, 0] * data["M_real"]

    lambda_rep = np.zeros((n_draws, N_real), dtype=np.float32)
    y_rep = np.zeros((n_draws, N_real), dtype=np.float32)

    for i in range(n_draws):
        b0 = float(beta_samples[i])
        rho_val = float(np.clip(rho_samples[i], 1e-4, 1.0 - 1e-4))
        eta_i = float(eta_samples[i])

        W_binary = (data["Z_real"] * eta_i <= THRESHOLD).astype(np.float32)
        A_filtered_i = data["A_real"] * W_binary
        A_filtered_i = _repair_isolates_deterministic(A_filtered_i, data["A_real"], data["Z_real"])

        w_empirical = np.log(data["y_real"] + 0.5) - np.log(data["e_real"]) - b0
        w_empirical = (w_empirical - w_empirical.mean()).astype(np.float32)

        ordering = _get_ordering(N_real, mode=ordering_mode)
        ImB, _ = _dagar_factors(A_filtered_i, rho_val, ordering)
        w_smoothed = np.linalg.solve(ImB, w_empirical).astype(np.float32)
        w_smoothed = w_smoothed - w_smoothed.mean()

        log_poisson_lam = np.log(data["e_real"]) + b0 + w_smoothed
        log_poisson_lam = np.clip(log_poisson_lam, LOG_PREDICTIVE_LAMBDA_MIN, LOG_PREDICTIVE_LAMBDA_MAX)
        poisson_lam = np.exp(log_poisson_lam).astype(np.float32)
        lambda_rep[i, :] = poisson_lam
        y_rep[i, :] = np.random.poisson(poisson_lam)

    return {
        "fitted_count_mean": lambda_rep.mean(axis=0),
        "pred_count_mean": y_rep.mean(axis=0),
        "pred_count_lower": np.quantile(y_rep, 0.025, axis=0),
        "pred_count_upper": np.quantile(y_rep, 0.975, axis=0),
    }


def compute_abi_script_matched_plot_summaries(post_draws, data, num_samples=SCRIPT_PLOT_NUM_SAMPLES):
    n_draws = min(num_samples, post_draws["beta"].shape[1])
    N_real = data["N_real"]
    threshold = np.log(2.0)

    beta_samples = post_draws["beta"][0, :n_draws, 0]
    rho_samples = post_draws["rho"][0, :n_draws, 0]
    eta_samples = post_draws["eta_raw"][0, :n_draws, 0] * data["M_real"]

    y_rep = np.zeros((n_draws, N_real), dtype=np.float32)

    for i in range(n_draws):
        b0 = float(beta_samples[i])
        rho_val = float(np.clip(rho_samples[i], 1e-4, 1.0 - 1e-4))
        eta_i = float(eta_samples[i])

        W_binary = (data["Z_real"] * eta_i <= threshold).astype(np.float32)
        A_filtered_i = data["A_real"] * W_binary

        for node in range(N_real):
            if A_filtered_i[node].sum() == 0:
                neighbors = np.where(data["A_real"][node] == 1)[0]
                if len(neighbors) > 0:
                    j = np.random.choice(neighbors)
                    A_filtered_i[node, j] = 1.0
                    A_filtered_i[j, node] = 1.0

        w_empirical = np.log(data["y_real"] + 0.1) - b0 - np.log(data["e_real"])
        ordering = np.random.permutation(N_real).astype(int)
        ImB, _ = _dagar_factors(A_filtered_i, rho_val, ordering)
        w_smoothed = np.linalg.solve(ImB, w_empirical)

        log_poisson_lam = np.log(data["e_real"]) + b0 + w_smoothed
        log_poisson_lam = np.clip(log_poisson_lam, LOG_PREDICTIVE_LAMBDA_MIN, LOG_PREDICTIVE_LAMBDA_MAX)
        poisson_lam = np.exp(log_poisson_lam).astype(np.float32)
        y_rep[i, :] = np.random.poisson(poisson_lam)

    y_rep_mean = y_rep.mean(axis=0)
    return {
        "observed": data["y_real"].copy(),
        "expected": data["e_real"].copy(),
        "pred_count_mean": y_rep_mean,
        "pred_count_lower": np.percentile(y_rep, 2.5, axis=0),
        "pred_count_upper": np.percentile(y_rep, 97.5, axis=0),
        "fitted_risk": y_rep_mean / data["e_real"],
    }


def load_carbayes_outputs(dataset_name):
    out_dir = CARBAYES_DIR / dataset_name
    if not out_dir.exists():
        raise FileNotFoundError(
            f"Missing CARBayes exports for {dataset_name}. Run CARBayes_california_glasgow_south_korea.R first."
        )

    return {
        "summary": pd.read_csv(out_dir / f"{dataset_name}_posterior_summary.csv"),
        "samples": pd.read_csv(out_dir / f"{dataset_name}_posterior_samples.csv"),
        "areas": pd.read_csv(out_dir / f"{dataset_name}_area_table.csv", dtype={"area_id": str}),
        "edges": pd.read_csv(out_dir / f"{dataset_name}_edge_table.csv", dtype={"from_id": str, "to_id": str}),
    }


def summarize_risk_comparison(area_compare):
    x = area_compare["risk_abi"].to_numpy()
    y = area_compare["risk_carbayes"].to_numpy()
    slope, intercept = np.polyfit(y, x, 1)
    return pd.DataFrame([
        {"metric": "correlation", "value": float(np.corrcoef(x, y)[0, 1])},
        {"metric": "mae", "value": float(np.mean(np.abs(x - y)))},
        {"metric": "rmse", "value": float(np.sqrt(np.mean((x - y) ** 2)))},
        {"metric": "abi_on_car_slope", "value": float(slope)},
        {"metric": "abi_on_car_intercept", "value": float(intercept)},
    ])


def summarize_edge_comparison(edge_compare):
    x = edge_compare["boundary_prob_abi"].to_numpy()
    y = edge_compare["boundary_prob"].to_numpy()
    z = edge_compare["edge_dissimilarity"].to_numpy()
    sel_abi = edge_compare["boundary_median_abi"].to_numpy().astype(bool)
    sel_car = edge_compare["boundary_median_car"].to_numpy().astype(bool)

    intersection = int(np.sum(sel_abi & sel_car))
    union = int(np.sum(sel_abi | sel_car))
    jaccard = intersection / union if union > 0 else np.nan

    return pd.DataFrame([
        {"metric": "boundary_prob_correlation", "value": float(np.corrcoef(x, y)[0, 1])},
        {"metric": "boundary_prob_mae", "value": float(np.mean(np.abs(x - y)))},
        {"metric": "abi_prob_vs_dissimilarity_corr", "value": float(np.corrcoef(z, x)[0, 1])},
        {"metric": "carbayes_prob_vs_dissimilarity_corr", "value": float(np.corrcoef(z, y)[0, 1])},
        {"metric": "median_boundary_count_abi", "value": int(sel_abi.sum())},
        {"metric": "median_boundary_count_carbayes", "value": int(sel_car.sum())},
        {"metric": "median_boundary_overlap", "value": intersection},
        {"metric": "median_boundary_jaccard", "value": float(jaccard)},
    ])


def _binned_probability_curve(x, y, n_bins=12):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    bins = np.unique(np.quantile(x, quantiles))
    if bins.size < 2:
        return x, y

    centers = []
    means = []
    for left, right in zip(bins[:-1], bins[1:]):
        if right == bins[-1]:
            mask = (x >= left) & (x <= right)
        else:
            mask = (x >= left) & (x < right)
        if np.any(mask):
            centers.append(float(x[mask].mean()))
            means.append(float(y[mask].mean()))

    return np.asarray(centers), np.asarray(means)


MAP_FIGSIZE = (12, 12)


def style_boxed_axes(ax):
    ax.set_facecolor("white")
    ax.set_axisbelow(True)
    ax.grid(True, color="#b0b0b0", linewidth=0.8, alpha=0.35)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#262626")
        spine.set_linewidth(0.8)


def plot_risk_comparison(area_compare, title=None):
    x = area_compare["risk_carbayes"].to_numpy()
    y = area_compare["risk_abi"].to_numpy()
    slope, intercept = np.polyfit(x, y, 1)

    lo = float(min(x.min(), y.min()))
    hi = float(max(x.max(), y.max()))
    grid = np.linspace(lo, hi, 200)

    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.scatter(x, y, alpha=0.7)
    ax.plot(grid, grid, linestyle="--", color="black", linewidth=1.2, label="45-degree line")
    ax.plot(grid, slope * grid + intercept, color="firebrick", linewidth=2.0, label="Least-squares fit")
    ax.set_xlabel("CARBayes fitted risk")
    ax.set_ylabel("ABI fitted risk")
    ax.legend(frameon=False)
    style_boxed_axes(ax)
    return fig, ax


def plot_posterior_predictive_check(abi_script_plots, title=None):
    y_obs = abi_script_plots["observed"]
    y_mean = abi_script_plots["pred_count_mean"]
    y_lower = abi_script_plots["pred_count_lower"]
    y_upper = abi_script_plots["pred_count_upper"]

    fig, ax = plt.subplots(1, 1, figsize=(7, 6))
    ax.errorbar(
        y_obs,
        y_mean,
        yerr=[y_mean - y_lower, y_upper - y_mean],
        fmt="o",
        alpha=0.7,
        ecolor="lightgray",
        capsize=3,
        label="Predicted mean (95% CI)",
    )

    max_val = float(max(y_obs.max(), y_upper.max()))
    ax.plot([0, max_val], [0, max_val], "r--", linewidth=2, label="Perfect fit (y = x)")
    ax.set_xlabel("Observed counts")
    ax.set_ylabel("Predicted counts")
    ax.legend(frameon=False)
    style_boxed_axes(ax)
    return fig, ax


def plot_edge_probability_vs_dissimilarity(edge_compare, title=None):
    z = edge_compare["edge_dissimilarity"].to_numpy()
    y_car = edge_compare["boundary_prob"].to_numpy()
    y_abi = edge_compare["boundary_prob_abi"].to_numpy()

    z_car_curve, car_curve = _binned_probability_curve(z, y_car)
    z_abi_curve, abi_curve = _binned_probability_curve(z, y_abi)

    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.scatter(z, y_car, alpha=0.18, s=18, color="purple")
    ax.scatter(z, y_abi, alpha=0.18, s=18, color="darkgreen")
    ax.plot(z_car_curve, car_curve, color="purple", marker="o", linewidth=2.0, label="CARBayes")
    ax.plot(z_abi_curve, abi_curve, color="darkgreen", marker="o", linewidth=2.0, label="ABI")
    ax.set_xlabel("Standardized edge dissimilarity |x_i - x_j|")
    ax.set_ylabel("Posterior boundary probability")
    ax.legend(frameon=False)
    style_boxed_axes(ax)
    return fig, ax


def build_boundary_geometries(gdf, edge_subset, id_col):
    id_to_pos = {str(area_id): idx for idx, area_id in enumerate(gdf[id_col].astype(str))}
    geoms = []

    for _, row in edge_subset.iterrows():
        i = id_to_pos[str(row["from_id"])]
        j = id_to_pos[str(row["to_id"])]
        shared = gdf.iloc[i].geometry.intersection(gdf.iloc[j].geometry)
        shared_line = extract_line_geometry(shared)
        if shared_line is not None and not shared_line.is_empty:
            geoms.append(shared_line)

    if len(geoms) == 0:
        return None

    return gpd.GeoDataFrame({"geometry": geoms}, crs=gdf.crs)


def extract_line_geometry(shared):
    if shared.is_empty:
        return None
    if shared.geom_type in ["LineString", "MultiLineString"]:
        return shared
    if shared.geom_type in ["Polygon", "MultiPolygon"]:
        return shared.boundary
    if shared.geom_type == "GeometryCollection":
        from shapely.ops import unary_union

        lines = []
        for geom in shared.geoms:
            if geom.geom_type in ["LineString", "MultiLineString"]:
                lines.append(geom)
            elif geom.geom_type in ["Polygon", "MultiPolygon"]:
                lines.append(geom.boundary)
        if len(lines) > 0:
            return unary_union(lines)
    return None


def build_boundary_probability_geometries(gdf, edge_table, id_col):
    id_to_pos = {str(area_id): idx for idx, area_id in enumerate(gdf[id_col].astype(str))}
    records = []

    for _, row in edge_table.iterrows():
        from_id = str(row["from_id"])
        to_id = str(row["to_id"])
        if from_id not in id_to_pos or to_id not in id_to_pos:
            continue
        i = id_to_pos[from_id]
        j = id_to_pos[to_id]
        geom_i = gdf.iloc[i].geometry
        geom_j = gdf.iloc[j].geometry
        try:
            shared = geom_i.intersection(geom_j)
        except Exception:
            shared = geom_i.buffer(0).intersection(geom_j.buffer(0))
        shared_line = extract_line_geometry(shared)
        if shared_line is not None:
            records.append({"geometry": shared_line, "boundary_prob_abi": float(row["boundary_prob_abi"])})

    if len(records) == 0:
        return None

    return gpd.GeoDataFrame(records, crs=gdf.crs)


def get_plot_gdf(gdf):
    if gdf.crs is None or not getattr(gdf.crs, "is_geographic", False):
        return gdf

    try:
        plot_crs = gdf.estimate_utm_crs()
        if plot_crs is None:
            return gdf
        return gdf.to_crs(plot_crs)
    except Exception:
        return gdf


def save_figure(fig, path, tight=True):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    bbox_inches = "tight" if tight else None
    save_kwargs = {"dpi": 300, "bbox_inches": bbox_inches}
    if tight:
        save_kwargs["pad_inches"] = 0.02
    fig.savefig(path, **save_kwargs)


def style_map_legend(fig):
    if len(fig.axes) < 2:
        return

    cax = fig.axes[-1]
    cax.tick_params(labelsize=MAP_LEGEND_TICKSIZE)
    cax.yaxis.label.set_size(MAP_LEGEND_LABELSIZE)


def plot_boundary_probability_map(data, edge_compare, config, title=None):
    gdf_plot = get_plot_gdf(data["gdf"]).copy()
    risk_col = config.get("risk_map_col")
    if risk_col is not None and risk_col in gdf_plot.columns:
        gdf_plot["map_observed_risk"] = np.asarray(gdf_plot[risk_col], dtype=float)
    else:
        gdf_plot["map_observed_risk"] = np.asarray(data["y_real"] / data["e_real"], dtype=float)

    risk_vmin = float(np.nanmin(gdf_plot["map_observed_risk"]))
    risk_vmax = float(np.nanmax(gdf_plot["map_observed_risk"]))
    edge_geoms = build_boundary_probability_geometries(gdf_plot, edge_compare, config["id_col"])

    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE)
    gdf_plot.plot(
        column="map_observed_risk",
        ax=ax,
        cmap="Greys",
        edgecolor="white",
        linewidth=0.35,
        alpha=0.75,
        vmin=risk_vmin,
        vmax=risk_vmax,
        legend=False,
    )

    if edge_geoms is not None:
        edge_geoms = edge_geoms.sort_values("boundary_prob_abi")
        from matplotlib.collections import LineCollection
        from matplotlib.colors import Normalize

        segments = []
        probabilities = []
        for _, row in edge_geoms.iterrows():
            geom = row.geometry
            line_parts = list(geom.geoms) if geom.geom_type == "MultiLineString" else [geom]
            for line in line_parts:
                if line.geom_type == "LineString":
                    segments.append(np.asarray(line.coords))
                    probabilities.append(float(row["boundary_prob_abi"]))

        if len(segments) > 0:
            probabilities = np.asarray(probabilities, dtype=float)
            edge_collection = LineCollection(
                segments,
                cmap="viridis",
                norm=Normalize(vmin=0.0, vmax=1.0),
                zorder=3,
            )
            edge_collection.set_array(probabilities)
            edge_collection.set_linewidths(0.35 + 3.25 * probabilities)
            edge_collection.set_capstyle("round")
            ax.add_collection(edge_collection)

            cbar = fig.colorbar(edge_collection, ax=ax, fraction=0.035, pad=0.015, shrink=MAP_LEGEND_SHRINK)
            cbar.set_label("ABI posterior boundary probability")
            cbar.ax.tick_params(labelsize=MAP_LEGEND_TICKSIZE)
            cbar.ax.yaxis.label.set_size(MAP_LEGEND_LABELSIZE)

    ax.axis("off")
    return fig, ax


def plot_boundary_agreement(data, edge_compare, config, title=None):
    both = edge_compare[(edge_compare["boundary_median_abi"] == 1) & (edge_compare["boundary_median_car"] == 1)]
    abi_only = edge_compare[(edge_compare["boundary_median_abi"] == 1) & (edge_compare["boundary_median_car"] == 0)]
    car_only = edge_compare[(edge_compare["boundary_median_abi"] == 0) & (edge_compare["boundary_median_car"] == 1)]

    gdf_plot = get_plot_gdf(data["gdf"]).copy()
    gdf_plot["map_fitted_risk_abi"] = np.asarray(data["fitted_risk_abi_for_map"], dtype=float)
    map_vmin = float(np.nanmin(gdf_plot["map_fitted_risk_abi"]))
    map_vmax = float(np.nanmax(gdf_plot["map_fitted_risk_abi"]))

    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE)
    gdf_plot.plot(
        column="map_fitted_risk_abi",
        ax=ax,
        cmap="YlOrRd",
        edgecolor="grey",
        linewidth=0.5,
        alpha=1.0,
        vmin=map_vmin,
        vmax=map_vmax,
        legend=True,
        legend_kwds={"label": "ABI fitted risk", "shrink": MAP_LEGEND_SHRINK},
    )
    style_map_legend(fig)

    geoms_both = build_boundary_geometries(gdf_plot, both, config["id_col"])
    geoms_abi = build_boundary_geometries(gdf_plot, abi_only, config["id_col"])
    geoms_car = build_boundary_geometries(gdf_plot, car_only, config["id_col"])

    if geoms_both is not None:
        geoms_both.plot(ax=ax, color="blue", linewidth=MAP_BOUNDARY_LINEWIDTH)
    if geoms_abi is not None:
        geoms_abi.plot(ax=ax, color="darkgreen", linewidth=MAP_BOUNDARY_LINEWIDTH)
    if geoms_car is not None:
        geoms_car.plot(ax=ax, color="purple", linewidth=MAP_BOUNDARY_LINEWIDTH)

    ax.axis("off")
    return fig, ax


def plot_fitted_risk_surface(data, abi_script_plots, abi_edges, config, title=None):
    gdf_plot = get_plot_gdf(data["gdf"]).copy()
    gdf_plot["fitted_risk_abi"] = abi_script_plots["fitted_risk"]
    gdf_plot["map_fitted_risk_abi"] = np.asarray(abi_script_plots["fitted_risk"], dtype=float)
    map_vmin = float(np.nanmin(gdf_plot["map_fitted_risk_abi"]))
    map_vmax = float(np.nanmax(gdf_plot["map_fitted_risk_abi"]))

    boundary_subset = abi_edges[abi_edges["boundary_median_abi"] == 1].copy()
    boundary_geoms = build_boundary_geometries(gdf_plot, boundary_subset, config["id_col"])

    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE)
    gdf_plot.plot(
        column="map_fitted_risk_abi",
        ax=ax,
        cmap="YlOrRd",
        edgecolor="grey",
        linewidth=0.5,
        alpha=1.0,
        vmin=map_vmin,
        vmax=map_vmax,
        legend=True,
        legend_kwds={"label": "ABI fitted risk", "shrink": MAP_LEGEND_SHRINK},
    )
    style_map_legend(fig)

    if boundary_geoms is not None:
        boundary_geoms.plot(ax=ax, color="blue", linewidth=MAP_BOUNDARY_LINEWIDTH)

    ax.axis("off")
    return fig, ax


def run_application(workflow, config):
    data = prepare_real_dataset(config)
    post_draws = workflow.sample(conditions=data["conditions"], num_samples=ABI_NUM_SAMPLES)

    abi_summary = extract_abi_summary(post_draws, data["M_real"])
    abi_edges = compute_abi_edge_table(post_draws, data["x_real"], data["A_real"], data["area_ids"], data["M_real"])
    abi_predictive = compute_abi_predictive_summaries(post_draws, data)
    abi_script_plots = compute_abi_script_matched_plot_summaries(post_draws, data)
    data["fitted_risk_abi_for_map"] = np.asarray(abi_script_plots["fitted_risk"], dtype=float)

    abi_areas = pd.DataFrame({
        "area_id": data["area_ids"],
        "risk_abi": abi_predictive["fitted_count_mean"] / data["e_real"],
        "fitted_count_mean_abi": abi_predictive["fitted_count_mean"],
        "pred_count_mean_abi": abi_predictive["pred_count_mean"],
        "pred_count_lower_abi": abi_predictive["pred_count_lower"],
        "pred_count_upper_abi": abi_predictive["pred_count_upper"],
        "observed": data["y_real"],
        "expected": data["e_real"],
    })

    car = load_carbayes_outputs(config["name"])
    car_summary = car["summary"].copy()
    car_edges = car["edges"].copy().rename(columns={"boundary_median": "boundary_median_car"})
    car_areas = car["areas"].copy()

    summary_compare = build_parameter_comparison(abi_summary, car_summary)

    area_compare = abi_areas.merge(car_areas[["area_id", "risk_carbayes"]], on="area_id", how="inner")
    edge_compare = abi_edges.merge(
        car_edges[["from_id", "to_id", "boundary_prob", "boundary_median_car"]],
        on=["from_id", "to_id"],
        how="inner"
    )

    risk_metrics = summarize_risk_comparison(area_compare)
    edge_metrics = summarize_edge_comparison(edge_compare)

    output_dir = COMPARISON_DIR / config["name"]
    output_dir.mkdir(parents=True, exist_ok=True)
    abi_summary.to_csv(output_dir / "abi_posterior_summary.csv", index=False)
    summary_compare.to_csv(output_dir / "parameter_summary_comparison.csv", index=False)
    area_compare.to_csv(output_dir / "risk_comparison.csv", index=False)
    edge_compare.to_csv(output_dir / "edge_comparison.csv", index=False)
    risk_metrics.to_csv(output_dir / "risk_metrics.csv", index=False)
    edge_metrics.to_csv(output_dir / "edge_metrics.csv", index=False)

    return {
        "data": data,
        "post_draws": post_draws,
        "abi_summary": abi_summary,
        "abi_areas": abi_areas,
        "abi_edges": abi_edges,
        "summary_compare": summary_compare,
        "area_compare": area_compare,
        "edge_compare": edge_compare,
        "risk_metrics": risk_metrics,
        "edge_metrics": edge_metrics,
        "output_dir": output_dir,
        "abi_script_plots": abi_script_plots,
    }


In [ ]:
workflow = keras.saving.load_model(str(MODEL_PATH))

APPLICATIONS = {
    "glasgow": {
        "name": "glasgow",
        "gpkg": "respiratory_data_glasgow.gpkg",
        "adjacency_csv": "adjacency_matrix_glasgow.csv",
        "id_col": "IZ",
        "x_col": "incomedep",
        "y_col": "observed",
        "e_col": "expected",
        "risk_map_col": "SMR" if "SMR" in gpd.read_file(DATA_DIR / "respiratory_data_glasgow.gpkg").columns else "observed",
    },
    "california": {
        "name": "california",
        "gpkg": "respiratory_data_california.gpkg",
        "adjacency_csv": "adjacency_matrix_california.csv",
        "id_col": "county",
        "x_col": "smoking",
        "y_col": "lung_O_count",
        "e_col": "lung_E_count",
        "risk_map_col": "lung_standard_ratio",
    },
    "south_korea": {
        "name": "south_korea",
        "gpkg": "South_Korea/mortality_data_south_korea.gpkg",
        "adjacency_csv": "South_Korea/adjacency_matrix_south_korea.csv",
        "id_col": "area_id",
        "x_col": "smoking_pct",
        "y_col": "observed_lung_cancer",
        "e_col": "expected_lung_cancer",
        "risk_map_col": "smr_lung_cancer",
    },
}

print("Loaded ABI model and application configurations.")

In [ ]:
glasgow = run_application(workflow, APPLICATIONS["glasgow"])

print("Glasgow: posterior summary comparison")
display(glasgow["summary_compare"])

print("Glasgow: risk comparison metrics")
display(glasgow["risk_metrics"])

print("Glasgow: edge comparison metrics")
display(glasgow["edge_metrics"])

fig, _ = plot_risk_comparison(glasgow["area_compare"], "Glasgow risk comparison")
save_figure(fig, glasgow["output_dir"] / "glasgow_risk_comparison.png")
plt.show()

fig, _ = plot_posterior_predictive_check(
    glasgow["abi_script_plots"],
    "Glasgow posterior predictive check: node-level counts vs predictions"
)
save_figure(fig, glasgow["output_dir"] / "glasgow_post_pred_check.png")
plt.show()

fig, _ = plot_edge_probability_vs_dissimilarity(
    glasgow["edge_compare"],
    "Glasgow boundary probability versus edge dissimilarity"
)
save_figure(fig, glasgow["output_dir"] / "glasgow_edge_probability_vs_dissimilarity.png")
plt.show()

fig, _ = plot_boundary_probability_map(
    glasgow["data"],
    glasgow["edge_compare"],
    APPLICATIONS["glasgow"],
    title="Glasgow ABI posterior boundary probabilities"
)
save_figure(fig, glasgow["output_dir"] / "glasgow_boundary_probability_map.png")
plt.show()


fig, _ = plot_boundary_agreement(
    glasgow["data"],
    glasgow["edge_compare"],
    APPLICATIONS["glasgow"],
    title="Glasgow: median-probability boundary agreement (blue=both, green=ABI only, purple=CARBayes only)"
)
save_figure(fig, glasgow["output_dir"] / "glasgow_boundary_agreement.png")
plt.show()

In [ ]:
california = run_application(workflow, APPLICATIONS["california"])

print("California: posterior summary comparison")
display(california["summary_compare"])

print("California: risk comparison metrics")
display(california["risk_metrics"])

print("California: edge comparison metrics")
display(california["edge_metrics"])

fig, _ = plot_risk_comparison(california["area_compare"], "California risk comparison")
save_figure(fig, california["output_dir"] / "california_risk_comparison.png")
plt.show()

fig, _ = plot_posterior_predictive_check(
    california["abi_script_plots"],
    "California posterior predictive check: node-level counts vs predictions"
)
save_figure(fig, california["output_dir"] / "california_post_pred_check.png")
plt.show()

fig, _ = plot_edge_probability_vs_dissimilarity(
    california["edge_compare"],
    "California boundary probability versus edge dissimilarity"
)
save_figure(fig, california["output_dir"] / "california_edge_probability_vs_dissimilarity.png")
plt.show()

fig, _ = plot_boundary_probability_map(
    california["data"],
    california["edge_compare"],
    APPLICATIONS["california"],
    title="California ABI posterior boundary probabilities"
)
save_figure(fig, california["output_dir"] / "california_boundary_probability_map.png")
plt.show()


fig, _ = plot_boundary_agreement(
    california["data"],
    california["edge_compare"],
    APPLICATIONS["california"],
    title="California: median-probability boundary agreement (blue=both, green=ABI only, purple=CARBayes only)"
)
save_figure(fig, california["output_dir"] / "california_boundary_agreement.png")
plt.show()

In [ ]:
south_korea = run_application(workflow, APPLICATIONS["south_korea"])

print("South Korea lung cancer: posterior summary comparison")
display(south_korea["summary_compare"])

print("South Korea lung cancer: risk comparison metrics")
display(south_korea["risk_metrics"])

print("South Korea lung cancer: edge comparison metrics")
display(south_korea["edge_metrics"])

fig, _ = plot_risk_comparison(south_korea["area_compare"], "South Korea lung-cancer risk comparison")
save_figure(fig, south_korea["output_dir"] / "south_korea_risk_comparison.png")
plt.show()

fig, _ = plot_posterior_predictive_check(
    south_korea["abi_script_plots"],
    "South Korea lung-cancer posterior predictive check: node-level counts vs predictions"
)
save_figure(fig, south_korea["output_dir"] / "south_korea_post_pred_check.png")
plt.show()

fig, _ = plot_edge_probability_vs_dissimilarity(
    south_korea["edge_compare"],
    "South Korea lung-cancer boundary probability versus edge dissimilarity"
)
save_figure(fig, south_korea["output_dir"] / "south_korea_edge_probability_vs_dissimilarity.png")
plt.show()

fig, _ = plot_boundary_probability_map(
    south_korea["data"],
    south_korea["edge_compare"],
    APPLICATIONS["south_korea"],
    title="South Korea ABI posterior boundary probabilities"
)
save_figure(fig, south_korea["output_dir"] / "south_korea_boundary_probability_map.png")
plt.show()


fig, _ = plot_boundary_agreement(
    south_korea["data"],
    south_korea["edge_compare"],
    APPLICATIONS["south_korea"],
    title="South Korea: median-probability boundary agreement (blue=both, green=ABI only, purple=CARBayes only)"
)
save_figure(fig, south_korea["output_dir"] / "south_korea_boundary_agreement.png")
plt.show()

In [ ]:
application_results = {"glasgow": glasgow, "california": california, "south_korea": south_korea}

combined_risk_metrics = pd.concat([
    result["risk_metrics"].assign(dataset=dataset_name)
    for dataset_name, result in application_results.items()
], ignore_index=True)

combined_edge_metrics = pd.concat([
    result["edge_metrics"].assign(dataset=dataset_name)
    for dataset_name, result in application_results.items()
], ignore_index=True)

print("Combined risk metrics")
display(combined_risk_metrics)

print("Combined edge metrics")
display(combined_edge_metrics)

combined_risk_metrics.to_csv(COMPARISON_DIR / "combined_risk_metrics.csv", index=False)
combined_edge_metrics.to_csv(COMPARISON_DIR / "combined_edge_metrics.csv", index=False)
print("Saved comparison outputs under:", COMPARISON_DIR)

In [ ]:
from PIL import Image


MAP_PAIRS = [
    (
        glasgow["output_dir"] / "glasgow_boundary_agreement.png",
        california["output_dir"] / "california_boundary_agreement.png",
    ),
    (
        glasgow["output_dir"] / "glasgow_boundary_agreement.png",
        south_korea["output_dir"] / "south_korea_boundary_agreement.png",
    ),
]


def _content_bbox(image, tolerance=8):
    arr = np.array(image.convert("RGBA"))
    bg = arr[0, 0].astype(int)
    diff = np.max(np.abs(arr.astype(int) - bg), axis=2)
    mask = diff > tolerance
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return (0, 0, image.width, image.height)
    return (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)


def _column_clusters(mask, min_pixels=10):
    counts = mask.sum(axis=0)
    cols = np.where(counts > min_pixels)[0]
    if len(cols) == 0:
        return []
    gaps = np.where(np.diff(cols) > 1)[0]
    starts = [cols[0]] + [cols[i + 1] for i in gaps]
    ends = [cols[i] for i in gaps] + [cols[-1]]
    return [(int(start), int(end) + 1) for start, end in zip(starts, ends)]


def _split_map_and_legend(image, tolerance=8, merge_gap=60):
    arr = np.array(image.convert("RGBA"))
    bg = arr[0, 0].astype(int)
    diff = np.max(np.abs(arr.astype(int) - bg), axis=2)
    mask = diff > tolerance
    clusters = _column_clusters(mask)
    if len(clusters) < 2:
        return image.crop(_content_bbox(image)), None

    legend_start, _ = clusters[-1]
    idx = len(clusters) - 2
    while idx >= 0 and legend_start - clusters[idx][1] <= merge_gap:
        legend_start = clusters[idx][0]
        idx -= 1

    if idx < 0:
        return image.crop(_content_bbox(image)), None

    map_part = image.crop((0, 0, legend_start, image.height))
    legend_part = image.crop((legend_start, 0, image.width, image.height))
    return map_part.crop(_content_bbox(map_part)), legend_part.crop(_content_bbox(legend_part))


def _resize_to_height(image, target_height):
    if image is None or image.height == target_height:
        return image
    new_width = int(round(image.width * target_height / image.height))
    return image.resize((new_width, target_height), Image.Resampling.LANCZOS)


def _match_visible_map_height(path_left, path_right, overwrite=True, gap_px=120, horizontal_padding_px=80):
    with Image.open(path_left) as left_img_raw, Image.open(path_right) as right_img_raw:
        left_img = left_img_raw.convert("RGBA")
        right_img = right_img_raw.convert("RGBA")

        left_map, left_legend = _split_map_and_legend(left_img)
        right_map, right_legend = _split_map_and_legend(right_img)

        target_map_height = max(left_map.height, right_map.height)
        left_map = _resize_to_height(left_map, target_map_height)
        right_map = _resize_to_height(right_map, target_map_height)

        legend_images = [image for image in (left_legend, right_legend) if image is not None]
        target_legend_height = int(round(MAP_LEGEND_SHRINK * target_map_height)) if legend_images else 0
        left_legend = _resize_to_height(left_legend, target_legend_height)
        right_legend = _resize_to_height(right_legend, target_legend_height)

        left_group_width = left_map.width + (gap_px if left_legend is not None else 0) + (left_legend.width if left_legend is not None else 0)
        right_group_width = right_map.width + (gap_px if right_legend is not None else 0) + (right_legend.width if right_legend is not None else 0)
        canvas_height = max(target_map_height, target_legend_height)
        background = tuple(int(x) for x in np.array(left_img)[0, 0])

        def _compose(map_image, legend_image):
            group_width = map_image.width + (gap_px if legend_image is not None else 0) + (legend_image.width if legend_image is not None else 0)
            canvas_width = group_width + 2 * horizontal_padding_px
            canvas = Image.new("RGBA", (canvas_width, canvas_height), background)
            x0 = horizontal_padding_px
            map_y = (canvas_height - map_image.height) // 2
            canvas.paste(map_image, (x0, map_y), map_image)
            if legend_image is not None:
                legend_x = x0 + map_image.width + gap_px
                legend_y = (canvas_height - legend_image.height) // 2
                canvas.paste(legend_image, (legend_x, legend_y), legend_image)
            return canvas

        left_final = _compose(left_map, left_legend)
        right_final = _compose(right_map, right_legend)

        if overwrite:
            left_out = path_left
            right_out = path_right
        else:
            left_out = path_left.with_stem(path_left.stem + "_same_height")
            right_out = path_right.with_stem(path_right.stem + "_same_height")

        left_final.save(left_out)
        right_final.save(right_out)

        print(
            f"Matched visible map height for {path_left.name} and {path_right.name}: "
            f"map height {target_map_height}px, legend height {target_legend_height}px, "
            f"canvas heights {canvas_height}px; widths {left_final.width}px and {right_final.width}px"
        )
        print(f"Saved: {left_out}")
        print(f"Saved: {right_out}")



def _match_target_to_reference_map_scale(reference_path, target_path, overwrite=True, gap_px=120, horizontal_padding_px=80):
    with Image.open(reference_path) as reference_raw, Image.open(target_path) as target_raw:
        reference_img = reference_raw.convert("RGBA")
        target_img = target_raw.convert("RGBA")

        reference_map, reference_legend = _split_map_and_legend(reference_img)
        target_map, target_legend = _split_map_and_legend(target_img)

        canvas_height = reference_img.height
        target_map = _resize_to_height(target_map, reference_map.height)
        if target_legend is not None:
            target_legend = _resize_to_height(
                target_legend,
                int(round(MAP_LEGEND_SHRINK * reference_map.height)),
            )

        background = tuple(int(x) for x in np.array(reference_img)[0, 0])
        group_width = target_map.width + (gap_px if target_legend is not None else 0) + (target_legend.width if target_legend is not None else 0)
        canvas_width = group_width + 2 * horizontal_padding_px
        canvas = Image.new("RGBA", (canvas_width, canvas_height), background)
        x0 = horizontal_padding_px
        map_y = (canvas_height - target_map.height) // 2
        canvas.paste(target_map, (x0, map_y), target_map)
        if target_legend is not None:
            legend_x = x0 + target_map.width + gap_px
            legend_y = (canvas_height - target_legend.height) // 2
            canvas.paste(target_legend, (legend_x, legend_y), target_legend)

        if overwrite:
            out_path = target_path
        else:
            out_path = target_path.with_stem(target_path.stem + "_same_scale")
        canvas.save(out_path)

        print(
            f"Matched {target_path.name} to reference scale of {reference_path.name}: "
            f"canvas {canvas_width}x{canvas_height}px, map height {reference_map.height}px"
        )
        print(f"Saved: {out_path}")


for left_path, right_path in MAP_PAIRS:
    _match_visible_map_height(left_path, right_path, overwrite=True)


MAP_SCALE_REFERENCES = [
    (
        glasgow["output_dir"] / "glasgow_boundary_agreement.png",
        glasgow["output_dir"] / "glasgow_boundary_probability_map.png",
    ),
    (
        california["output_dir"] / "california_boundary_agreement.png",
        california["output_dir"] / "california_boundary_probability_map.png",
    ),
    (
        south_korea["output_dir"] / "south_korea_boundary_agreement.png",
        south_korea["output_dir"] / "south_korea_boundary_probability_map.png",
    ),
]

for reference_path, target_path in MAP_SCALE_REFERENCES:
    _match_target_to_reference_map_scale(reference_path, target_path, overwrite=True)
